# Web Scraping with Python

*Author: Jemma Nelson*

First, import the packages we will need:

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from bs4 import BeautifulSoup
import bs4
import requests
# from selenium import webdriver 
# from selenium.webdriver.support.ui import Select


In [2]:
## Enable multiple outputs from jupyter cells
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
%matplotlib inline

> Note: In order to accomplish the final piece of this module (using Selenium), you must install a browser driver in addition to the package installations. You can find instructions [here](http://chromedriver.chromium.org/getting-started) for chromedriver. This module chose the `chromedriver,` but you can use others, such as drivers for Firefox or Safari.

## Web Scraping

### What is web scraping?

Web scraping is the process of collecting data from an HTML page or pages.

Many websites provide public-facing pages with continually updating data tables. Some sites have downloadable links for data (.csv or .xlsx files), but sometimes the data exist only on the page.

We could simply drag our cursor over the web page and use copy-and-paste, and hope that the data will flow into a spreadsheet program. Indeed, for one page or this might be the fastest way to capture the data.

But if there are many pages, or if we return to similar URLs over and over again for periodic data, wouldn't it be nice if there were a way to use our coding skills to automate the process? That's where web scraping comes in.

### The challenges of web scraping

The biggest challenges to capturing data from the web can be:

* HTML pages can be coded in any number of ways
* data may be present in a table, or in some other HTML structure
* data may be present with extraneous information (images, for example)
* data may exist on several pages
* we have to understand something of HTML architecture

### "Naive" approach: try pandas read_html()

If the data appear as a table on a page, we can point pandas' `read_html()` method to the page and hope for the best. Sometimes it works!

`read_html()` scans the page and looks for a `<table>` tag. It uses this tag to parse the data in the table.

Let's investigate this page from the US Bureau of Labor Statistics: https://www.bls.gov/eag/eag.us.htm

* returns a list of data frame objects, even if there is only one table

In [4]:
url = 'https://www.bls.gov/eag/eag.us.htm'
market_snapshots = pd.read_html(url)
#market_snapshots #list of data frame(s)

In [5]:
market_snapshots[0]

,Data Series,Back Data,May 2021,June 2021,July 2021,Aug 2021,Sept 2021,Oct 2021
0,Unemployment Rate(1),NaN,5.8,5.9,5.4,5.2,4.8,4.6
1,Change in Payroll Employment(2),NaN,614,962,1091,483,(p)312,(p)531
2,Average Hourly Earnings(3),NaN,30.31,30.44,30.55,30.67,(p)30.85,(p)30.96
3,Consumer Price Index(4),NaN,0.6,0.9,0.5,0.3,0.4,0.9
4,Producer Price Index(5),NaN,0.9,0.9,(p)0.7,(p)0.7,(p)0.5,(p)0.6
5,U.S. Import Price Index(6),NaN,1.3,1.1,0.3,(r)-0.2,(r)0.4,(r)1.2
6,"Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted...","Footnotes (1) In percent, seasonally adjusted..."


In [6]:
market_snapshots[0].info()
# We probably need to do additional cleanup. 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Data Series  7 non-null      object
 1   Back Data    1 non-null      object
 2   May 2021     7 non-null      object
 3   June 2021    7 non-null      object
 4   July 2021    7 non-null      object
 5   Aug 2021     7 non-null      object
 6   Sept 2021    7 non-null      object
 7   Oct 2021     7 non-null      object
dtypes: object(8)
memory usage: 576.0+ bytes


## What if it's not a table?

This website from the Colorado Health Department looks like a table, but actually isn't. Because of this, it fails to parse. 

https://cdphe.colorado.gov/workplace-safety/data-and-reports/colorado-health-indicators

By "table" we mean a specific html structure, a `<table>` tag. This is the what the Pandas command `read_html()` searches for in order to parse data into a dataframe.

Not everything that looks like a "table" on the web is an html table, which can be a challenge to programmatic scraping.

In [8]:
url = 'https://cdphe.colorado.gov/workplace-safety/data-and-reports/colorado-health-indicators'

# this does result in an error, we discuss why below.
df = pd.read_html(url)[0]


ValueError: No tables found

## More complexity requires HTML knowledge

Once we move away from simple tables, we need to understand a little bit about how web pages are structured.

Web pages are composed of 3 different technologies/languages:

* HTML
* Javascript
* CSS (Cascading Style Sheets)

### How a web page is structured

A web page's architecture is written in HTML.
HTML (HyperText Markup Language) is a tag-based scripting language.

Examples:

`<p>This is a paragraph</p>`  -- The `<p>` tag encloses a paragraph  

`This text will be <em>emphasized</em>` -- The `<em>`phasis tag specifies emphasis (italic or bold).

Tags control both the **layout** and the **styling** of a page.


In [9]:
%%html
<p>This is a paragraph</p>
This text will be <em>emphasized</em>

### The DOM

A webpage is layed out using HTML tags to populate the `DOM`, or [Document Object Model.](https://en.wikipedia.org/wiki/Document_Object_Model)

The DOM is a fancy way of saying the blueprint of the page. The DOM provides instructions on how all the pieces of the web page fit together: paragraphs, images, titles, sidebars, and hidden objects like code.

It does so through a tree structure. Each element of the page is a branch or node on the tree.

Most web pages consist of a `<head>` section, which usually contains scripting languages like JavaScript, and a `<body>` section, which contains all the text and images.

### Cascading Style Sheets

Although HTML has evolved over the years to provide more flexibility (currently we mostly use HTML5), and it provides some basic functionality for styling (`<em>` and `<font>` tags), it doesn't provide the full positional and styling control that designers need.

Cascading Style Sheets fill the gap. They provide a layer on top of HTML which gives more specific instruction on colors, fonts, weights, positions, layering, and animation. 

CSS works in concert with HTML. The layout of the page is generally written in HTML, and then CSS **attributes** and **selectors** target specific tags or classes of tags to describe how they should be displayed.

For example, CSS could specify that every other paragraph in a document should have a light grey background.

Or that headers should be large and pink. 

In [10]:
%%html
<style>
h3#foo {
    color: pink;
    font-size: 30pt !important;
}
p.special:nth-child(odd) {
    background: #CCC
}
</style>
<h3 id="foo">This is a level 3 header</h3>
<p class="special">This is <em>emphasized</em> code.</p>
<p class="special"> This is another paragraph.</p>
<p class="special"> Another</p>
<p class="special"> And another</p>

### CSS id and class

In the code above, we could have specified that every paragraph on this entire page was subject to the alternating color style (since Jupyter Notebooks are web pages also!)

But in order to keep the styling contained, we used two additional HTML/CSS selectors:

* **id** a unique identifier
* **class** a grouped identifier

**id** attributes must be unique to the page, in other words, they should not repeat. An id should refer to one and one element only.

**class** attributes can apply to groupings of tags. For example, a selection of paragraphs could all have a common style like a background color, or some radio buttons could all be colored green.

To effectively scrape pages, we will need to acquire a basic understanding of HTML tags and CSS selectors, and be comfortable looking "under the hood" at how web pages are constructed, since we want to target that underlying structure with our python code.

### Back to a page

Let's return to a web page and look "under the hood" to see how it's composed.

All modern web browsers have web inspectors, which allow you to interactively see the rendered page and its source. We will demonstrate with Chrome.

First, we'll make a small data table on *this* page, and then inspect it.

In [11]:
%%html
<style>
.province {
    font-weight: bold;
}
#ontario-row {
    color: red;
}
</style>
<table>
    <tr>
        <th>Year</th>
        <th>Province</th>
        <th>Measure</th>
    </tr>
    <tr>
        <td>2018</td>
        <td class="province">BC</td>
        <td>14.3</td>
    </tr>
    <tr>
        <td>2016</td>
        <td class="province">AB</td>
        <td>10.2</td>
    </tr>
    <tr id="ontario-row">
        <td>2013</td>
        <td class="province">ON</td>
        <td>16.9</td>
    </tr>
</table>
    

Year,Province,Measure
2018,BC,14.3
2016,AB,10.2
2013,ON,16.9


### Instructions: inspecting a page

* Hover your mouse over the rendered table.
* right-click and choose "Inspect"
* The inspector should open. 
* use the select tool to hover over elements of the page and see how the DOM expands to display the current element.

### Using BeautifulSoup

**BeautifulSoup** is the most common Python package used to scrape web pages.

It captures the HTML of a page, and then provides methods to help extract specific elements.

Here we are specifying a "page" as a string. 

We will then use beautiful soup to parse the page.

In [12]:
html_doc = """
<html><head><title>My Provincial Data</title>
<style>
.province {
    font-weight: bold;
}
#ontario-row {
    color: red;
}
</style>
</head>
<body>
<h3> This is a document providing some measurements.</h3>
<p> We could provide more detail here.</p>
<table>
    <tr>
        <th>Year</th>
        <th>Province</th>
        <th>Measure</th>
    </tr>
    <tr>
        <td>2018</td>
        <td class="province">BC</td>
        <td>14.3</td>
    </tr>
    <tr>
        <td>2016</td>
        <td class="province">AB</td>
        <td>10.2</td>
    </tr>
    <tr id="ontario-row">
        <td>2013</td>
        <td class="province">ON</td>
        <td>16.9</td>
    </tr>
</table>
<p> And maybe a summary here.</p>
"""

In [13]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_doc,'html.parser')
type(soup)


bs4.BeautifulSoup

In [14]:
soup.prettify()

'<html>\n <head>\n  <title>\n   My Provincial Data\n  </title>\n  <style>\n   .province {\n    font-weight: bold;\n}\n#ontario-row {\n    color: red;\n}\n  </style>\n </head>\n <body>\n  <h3>\n   This is a document providing some measurements.\n  </h3>\n  <p>\n   We could provide more detail here.\n  </p>\n  <table>\n   <tr>\n    <th>\n     Year\n    </th>\n    <th>\n     Province\n    </th>\n    <th>\n     Measure\n    </th>\n   </tr>\n   <tr>\n    <td>\n     2018\n    </td>\n    <td class="province">\n     BC\n    </td>\n    <td>\n     14.3\n    </td>\n   </tr>\n   <tr>\n    <td>\n     2016\n    </td>\n    <td class="province">\n     AB\n    </td>\n    <td>\n     10.2\n    </td>\n   </tr>\n   <tr id="ontario-row">\n    <td>\n     2013\n    </td>\n    <td class="province">\n     ON\n    </td>\n    <td>\n     16.9\n    </td>\n   </tr>\n  </table>\n  <p>\n   And maybe a summary here.\n  </p>\n </body>\n</html>'

### Extract pieces from the soup

You can now use object-oriented-like notation to extract pieces from the "soup."

- we can either search by an html tag, or find **all** instances of a tag.

In [15]:
soup.title

<title>My Provincial Data</title>

In [16]:
soup.title.string

'My Provincial Data'

In [17]:
soup.p

<p> We could provide more detail here.</p>

In [18]:
soup.p.string

' We could provide more detail here.'

In [19]:
soup.td

<td>2018</td>

In [20]:
soup.find_all('td')

[<td>2018</td>,
 <td class="province">BC</td>,
 <td>14.3</td>,
 <td>2016</td>,
 <td class="province">AB</td>,
 <td>10.2</td>,
 <td>2013</td>,
 <td class="province">ON</td>,
 <td>16.9</td>]

In [21]:
soup.find(id='ontario-row')

<tr id="ontario-row">
<td>2013</td>
<td class="province">ON</td>
<td>16.9</td>
</tr>

In [22]:
soup.find('table')

<table>
<tr>
<th>Year</th>
<th>Province</th>
<th>Measure</th>
</tr>
<tr>
<td>2018</td>
<td class="province">BC</td>
<td>14.3</td>
</tr>
<tr>
<td>2016</td>
<td class="province">AB</td>
<td>10.2</td>
</tr>
<tr id="ontario-row">
<td>2013</td>
<td class="province">ON</td>
<td>16.9</td>
</tr>
</table>

In [23]:
table = soup.find('table') #extract the table tag
pd.read_html(str(table))[0] #turn it into a string and then use pandas read_html

,Year,Province,Measure
0,2018,BC,14.3
1,2016,AB,10.2
2,2013,ON,16.9


### Another demo, a press release from the Colorado Health Department

Here's another website, a press release from the Colorado Health Department that has some useful information on it. We could copy and paste this info, but let's use it as example to explore how the package `beautiful soup` works. 

https://cdphe.colorado.gov/press-release/coloradans-urged-to-build-on-record-flu-vaccination-uptake-to-preserve-hospital

In [24]:
url = 'https://cdphe.colorado.gov/press-release/coloradans-urged-to-build-on-record-flu-vaccination-uptake-to-preserve-hospital'

### First attempt, read_html()

This will error with the "no tables found" message.

In [25]:
# prices = pd.read_html(url)
# prices[0] # no tables found

## Trying soup to extract watercolor prices

Hmm, that doesn't seem to be working. Let's try using beautiful soup to capture and extract part of the page.

In [26]:
import requests

page = requests.get(url) #This makes the connection to a live page
page.status_code #200 means we have successfully connected to the page
content = page.content
soup = BeautifulSoup(content)
soup.prettify()[:400] # limit output

200

'<!DOCTYPE html>\n<html dir="ltr" lang="en" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/200'

Next, we will look for all paragraph tags within the "soup."

In [27]:
soup.find_all('p')

[<p>Visit <a href="https://covid19.colorado.gov/for-coloradans/vaccine/where-can-i-get-vaccinated">"Where can I get vaccinated"</a> or call 1-877-COVAXCO (1-877-268-2926) for vaccine information.</p>,
 <p><strong>REMOTE, Nov. 16, 2021</strong> – More Coloradans than ever got their flu vaccine last year, a feat that state health officials are encouraging Coloradans to exceed this year to save lives and reduce stress on health care systems and workers who are responding to increased hospitalizations from COVID-19. </p>,
 <p>We encourage Coloradans to get their flu vaccine as soon as possible as flu cases typically start to increase in the fall and peak between December and February. As of November 16, seven people have been hospitalized due to the flu. </p>,
 <p>You can get the flu vaccine at the same time you get the COVID-19 vaccine or booster. Medicare, Medicaid, CHP+, and most private health insurers cover the full cost of the flu vaccine; you don’t have to pay anything to health car

## Combining this with string methods

Let's use our knowledge of string methods to extract all of the paragraphs that have this "County name (Number%)" structure. 

First we'll select the first 30 paragraphs, and from them only the ones that have contents (contents is not None) and in which the contents are a string. Because html structures can be quite complex, sometimes the contents of a tag is another set of tags. 

Next we will select just the strings where we detect a `%` symbol.

In [28]:
# Some of these paragraphs have other things inside of them. We just want the strings. 
[type(p.contents[0]) for p in soup.find_all('p')[:5]]

[bs4.element.NavigableString,
 bs4.element.Tag,
 bs4.element.NavigableString,
 bs4.element.NavigableString,
 bs4.element.NavigableString]

In [29]:
dta = [p.contents[0] for p in soup.find_all('p')[:30] if p.contents[0] is not None and \
 isinstance(p.contents[0],bs4.element.NavigableString)]
# select only the tags that contain a percent symbol
[d for d in dta if d.find('%') > -1]


['During the 2020-2021 season, the number of flu doses given across the state increased from the previous year by 11.1%. That means about 220,000 more Coloradans got the flu vaccine compared to the previous year.\xa0',
 'Sedgwick (103.8 %)',
 'Pitkin (32.8%)',
 'San Miguel (28.0%)',
 'Gunnison (23.6%)',
 'Huerfano (22.4%)',
 'Kit Karson (21.4%)',
 'Gilpin (21.2%)',
 'Ouray (21.2%)',
 'Summit (20.9%)\xa0',
 'Eagle (20.5%)',
 'Boulder (18.4%)\xa0',
 'La Plata (17.7%)',
 'Broomfield (17.5%)',
 'Garfield (15.8%)',
 'Denver (15.6%)',
 'San Juan (14.0%)',
 'Jefferson (13.5%)',
 'Clear Creek (13.3%)',
 'Lake (12.7%)',
 'Grand (12.7%)',
 'Custer (11.9%)',
 'El Paso (11.3%)']

Or, we could put our parsed results into a dataframe and do some pandas work from there. Let's try that.

In [30]:
df = pd.DataFrame({"pr_text":dta})
df

,pr_text
0,Visit
1,We encourage Coloradans to get their flu vacci...
2,You can get the flu vaccine at the same time y...
3,"“Flu seasons are unpredictable, but the flu va..."
4,"During the 2020-2021 season, the number of flu..."
5,Twenty-two counties saw an increase in the per...
6,Sedgwick (103.8 %)
7,Pitkin (32.8%)
8,San Miguel (28.0%)
9,Gunnison (23.6%)


In [31]:
flu_dosage_increase_2021 = df.iloc[6:28,:].copy()
flu_dosage_increase_2021.head()

,pr_text
6,Sedgwick (103.8 %)
7,Pitkin (32.8%)
8,San Miguel (28.0%)
9,Gunnison (23.6%)
10,Huerfano (22.4%)


In [32]:
# extract the word (\w+) just before the space and open parenthesis
flu_dosage_increase_2021['county'] = flu_dosage_increase_2021['pr_text'].str.extract('(\w+) \(')
flu_dosage_increase_2021.head()

,pr_text,county
6,Sedgwick (103.8 %),Sedgwick
7,Pitkin (32.8%),Pitkin
8,San Miguel (28.0%),Miguel
9,Gunnison (23.6%),Gunnison
10,Huerfano (22.4%),Huerfano


In [33]:
# extract the digits within the set of parentheses (excluding the percent sign)
flu_dosage_increase_2021['increase'] = flu_dosage_increase_2021['pr_text'].str.extract('\((\d+\.\d)\s*\%\)').astype('float')
flu_dosage_increase_2021.head()

,pr_text,county,increase
6,Sedgwick (103.8 %),Sedgwick,103.8
7,Pitkin (32.8%),Pitkin,32.8
8,San Miguel (28.0%),Miguel,28.0
9,Gunnison (23.6%),Gunnison,23.6
10,Huerfano (22.4%),Huerfano,22.4


In [34]:
flu_dosage_increase_2021.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 6 to 27
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pr_text   22 non-null     object 
 1   county    22 non-null     object 
 2   increase  22 non-null     float64
dtypes: float64(1), object(2)
memory usage: 660.0+ bytes


That worked well! But, it was pretty labor intensive for this small amount of data.

In practice, this kind of approach can work well for repetetive structures inside of web pages where a copy and paste is impractical, like product listings at a store.

### Further automation with selenium

These approaches work for a single URL, but what happens when we want to automate the browsing process?

If the web pages follow a convention in the url string, you can often use string code approaches.

For instance, if the url were (pseudo-code):

`http://my.website/jan-data`  
`http://my.website/feb-data`  
`http://my.website/mar-data`

You could create a list of `['jan','feb','mar']` and use string interpolation to build the url strings.


In [35]:
month_list = ['jan','feb','mar']
urls = [f'http://my.website/{m}-data' for m in month_list]
urls

['http://my.website/jan-data',
 'http://my.website/feb-data',
 'http://my.website/mar-data']

### Page interaction

But what if the sub-pages are not so straightforward? 

If they require some kind of interaction, such as clicking on interface elements?

Selenium is a package that can programmatically browse the web, so it can be used for this type of automation.

Let's recall the example from the Colorado Health Department the page that has downloadable information on it. Can we navigate to these tables and programatically click on them?

The installation for the selenium package is a little more complex than a typical Python package. In addition to the package itself, it needs a helper application to run called a webdriver. For this reason, we'll just demonstrate the approach here.

https://selenium-python.readthedocs.io/installation.html

https://selenium-python.readthedocs.io/installation.html#drivers



In [37]:
from selenium import webdriver 
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By

### Webdrivers

In order to have selenium control a driver, you will need to download and install a webdriver. Instructions from selenium can be found [here. ](https://selenium-python.readthedocs.io/installation.html#drivers)

First, we'll make sure that the drivers are working, and we'll navigate to www.google.com to make sure we can retrieve a page.

A "headless" browser application will open and move to this page. It may seem as if you have two browser apps running-- indeed you do. The second one is controlled by your python script. We'll show how this other browser changes as we control it from script. 

In [39]:
import time
from selenium import webdriver

driver = webdriver.Chrome('./chromedriver')
driver.get('http://www.google.com/')

time.sleep(5)

<ipython-input-39-6efe568a9f27>:6: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome('./chromedriver')


> For Windows users, you may have to provide the installation location for the chromedriver like so:
browser = webdriver.Chrome(r"C:\Windows\chromedriver_win32\chromedriver.exe")

We can navigate to another page. Now we'll navigate to a CDC page and look around.

In this url are commands (following the `?` that select page results). We've selected Environmental Health datasets here. 

We will wait a little bit after issuing the command to give the URL time to resolve.

In [40]:
url = 'https://data.cdc.gov/browse?category=Environmental+Health+%26+Toxicology&limitTo=datasets'
driver.get(url)
time.sleep(30)

Let's retrieve all the link elements on the page. By default, this page returns 10 links.

In [42]:
from selenium.webdriver.common.by import By
res = driver.find_elements(By.CLASS_NAME,"browse2-result-name-link")
#tabStoryPoint tab-widget tabFormatted
len(res)

10

We can "click" on one of these links by retrieving the object, and "clicking" on it. In the browser, this will navigate to that link.


In [43]:
res[0:2]
res[0].click()

[<selenium.webdriver.remote.webelement.WebElement (session="0b37ab74bcdcd18fbce233a91e5727aa", element="b3e992f9-145e-4e53-8dd2-c44166f1cc5d")>,
 <selenium.webdriver.remote.webelement.WebElement (session="0b37ab74bcdcd18fbce233a91e5727aa", element="363e80f8-8860-41e9-a7ff-fd40e51a7a84")>]

Ok, we've navigated to a new page. Now let's find the download button and try to click on that.

In [44]:
btn = driver.find_element(By.CLASS_NAME,"download")
btn.click()

Things are happening! That opened up another window in which we have to select the format for download. Let's try to target "CSV"

Once we click on this link, we will initiate a download.

In [45]:
csv_btn = driver.find_element(By.LINK_TEXT,"CSV")
csv_btn.click()

## Putting this all together.

How can this be useful? After all, we can navigate to this page and click on a link. 

But, using these methods we could navigate to all of these links and click on the download buttons. And that would save us time! (Although, we have to write the script...)

In the script below we cycle through the first two links, click on the download button and then the CSV buttons, and initiate downloads for the datasets. We could extend this to all ten links, but in the interests of time, we won't show that here. 

We have put some pauses in between. If you switch over to the browser being controlled by this script, you can see this process happening.


In [46]:
#for i in range(len(res)):
for i in range(2):   
    driver.get(url)
    time.sleep(5)
    res = driver.find_elements(By.CLASS_NAME,"browse2-result-name-link")
    #print(res)
    res[i].click()
    #time.sleep(5)
    btn = driver.find_element(By.CLASS_NAME,"download")
    btn.click()
    time.sleep(5)
    csv_btn = driver.find_element(By.LINK_TEXT,"CSV")
    csv_btn.click()
    time.sleep(10)

## Uses for programmatic browsing

This kind of programmatic browsing is most useful if there are clearly defined organizational structures in the pages your are targeting-- for instance, 100 links with information or weekly updates that you want to download. Then a script like this makes sense.

The challenge here is generalization-- although this particular script works well, another site might use a different structure. Or the development time and effort might outweigh the benefit.

## Conclusion


### The Legality of Web Scraping

- one more important note: check to make sure your web scraping is legal. In particular, a couple of landmark cases guide us that published data (such as on a web page) is legal to scrape. 
- commercial uses of web scraping is limited. Check with your legal department, if need be.

1: https://arstechnica.com/tech-policy/2019/09/web-scraping-doesnt-violate-anti-hacking-law-appeals-court-rules/

2 (Behind the Medium paywall): https://towardsdatascience.com/web-scraping-is-now-legal-6bf0e5730a78

### The Ethics of Web Scraping

Even if web scraping is legal, you want to do it in a non-harmful way. Consider for a moment: if you write a script that siphons data from 100 or 1000 pages, what happens to the server providing those pages?

Each page is a request, and that taxes the server. You could be "denying service" simply by making a lot of requests. In fact, there is a type of malicious attack on a web site called a ["Denial of Service"](https://en.wikipedia.org/wiki/Denial-of-service_attack)

In general, web sites will provide a `robots.txt` file that outlines what practices good citizens should follow when scraping a web site. Always check [`robots.txt`](https://www.oreilly.com/library/view/python-web-scraping/9781786462589/9ee32cdd-619b-4f4b-a812-f796276548e7.xhtml). To check the robots page, navigate to your website and type in robots.txt after the site name.  

You can use the requests module to specify headers that will [mollify `robots.txt`:](https://stackoverflow.com/questions/59183359/requests-beautifulsoup-vs-robots-txt)

### Wrap-up

This is a brief intro to web scraping with Python. 

You'll find many other examples on the web, in practice this is a complex topic because of the variety of ways that web pages can present data.

In general, by creatively combining the tools of Python, Pandas, BeautifulSoup, and Selenium, you should be able to capture most data from the web.

Happy scraping!